# 01 — Adoption ingest: Entra sign-in logs → Bronze JSON

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 12 — T5 (finished in the Sprint 12.1 mini-sprint) |
| **Layer** | `Files/Bronze/adoption/YYYY-MM-DD/signins.json` (raw JSON landing) |
| **Source** | Entra sign-in logs — `SigninLogs` in `log-ihzhhpf-sit` (Log Analytics), routed by `infra/modules/entra/adoption-telemetry.bicep` |
| **Target** | `lh_ihzhhpf_sit` — `Files/Bronze/adoption/<day>/signins.json` (one file per sign-in day) |
| **Contract** | `docs/superpowers/specs/2026-07-09-sprint-12-org-design.md` §7 |
| **Consumers** | `data-platform/notebooks/bva/ingest_bronze_adoption.py` (Sprint 15 BVA) |

Reads the last 24h of `ihzhhpf-app` sign-ins and lands them **verbatim as JSON**
under `Files/Bronze/adoption/<day>/signins.json`, one file per sign-in day. The
row shape is the Bronze adoption contract (§7) — **identical** to the synthetic
backfill produced by `data-platform/scripts/adoption_seed_synthetic.py`, so real
and seeded telemetry are interchangeable for the downstream BVA medallion. A
downstream **Silver** notebook does the deduplication + role/hospital join.

**No PHI** — sign-in metadata carries UPN + IP only; the IP is redacted to a
`/24` in Bronze. Scheduled nightly by `.github/workflows/adoption-refresh.yml`.

> **Source note.** This notebook reads from Log Analytics because
> `adoption-telemetry.bicep` already routes `SigninLogs` there. The same Bronze
> rows can be produced from the Microsoft Graph `auditLogs/signIns` endpoint
> (used read-only by the `onboarding-agent`): keep the same projection field
> names and the pure `adoption_transforms` mapping below is unchanged.

> When real telemetry has not yet accumulated, seed Bronze with
> `python3 data-platform/scripts/adoption_seed_synthetic.py --output-dir Files/Bronze --days 30`.


In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
target_lakehouse = 'lh_ihzhhpf_sit'
log_analytics_workspace = 'log-ihzhhpf-sit'  # Log Analytics workspace resource name (for logs / audit)
log_analytics_workspace_id = '48d02003-cc54-4888-9f39-4ec3cb03a5c7'  # LAW customerId (GUID) -- used by LogsQueryClient
bronze_root = 'Files/Bronze/adoption'        # lakehouse-relative Bronze landing folder
personas_path = 'Files/reference/personas.csv'  # UPN -> app_role roster (mirrors data/synthetic/personas.csv)
lookback_hours = 24                          # Log Analytics lookback window in hours
default_env = 'sit'                          # fallback slot when the app URL host is not -prod


## 1. Kusto query

Projects the raw sign-in fields the Bronze contract needs (design spec §7). The
`env` slot and the `appRole` join are derived below (env from the app host,
`appRole` from the personas roster); the Kusto projection keeps the raw shape.


In [ ]:
KQL = """
SigninLogs
| where AppDisplayName startswith "ihzhhpf-app"
| project
    TimeGenerated,
    UserId,
    UserPrincipalName,
    AppDisplayName,
    AppId,
    ResultType,
    IPAddress,
    ClientAppUsed,
    DeviceDetail_TrustType = tostring(DeviceDetail.trustType),
    Location_CountryOrRegion = tostring(LocationDetails.countryOrRegion)
"""
# Note: lookback window is applied via `timespan=timedelta(hours=lookback_hours)`
# in the LogsQueryClient call below -- not embedded in the KQL text.


## 2. Read from Log Analytics

Uses `azure-kusto-data.KustoClient` (installed via the `env-adoption-kusto`
Fabric environment) against Log Analytics' ADX-compatible query endpoint
(`https://ade.loganalytics.io/subscriptions/.../workspaces/<law>`). The auth
token comes from `notebookutils.credentials.getToken('kusto')` — the only
audience broker that Fabric supports for LAW-shaped queries. Log Analytics
Reader on `log-ihzhhpf-sit` is granted to the Fabric session identity by
`infra/modules/entra/adoption-telemetry.bicep`.

> **Fabric portability note.** This cell previously used the Synapse-only
> `spark.synapse.linkedService` option (via the Kusto Spark connector),
> which does not exist in Fabric (no linked-service concept). We also tried
> `azure-monitor-query` + `DefaultAzureCredential` — that path fails because
> Fabric notebooks don't support `DefaultAzureCredential` and the Fabric
> token broker rejects arbitrary resource URIs (only `storage`/`pbi`/
> `keyvault`/`kusto` audience keys work). The Kusto-client-through-LAW-ADE
> pattern below is the only Fabric-native path that runs end-to-end.
> Rewritten during the Sprint 11-16 fix pass on 2026-07-15.


In [ ]:
# --- Fabric-native Log Analytics query pattern ---
#
# 1. LAW's ADX-compatible query endpoint accepts 'kusto' audience tokens.
#    Cluster URL follows the ADE format:
#      https://ade.loganalytics.io/subscriptions/{sub}/resourcegroups/{rg}/
#        providers/microsoft.operationalinsights/workspaces/{name}
# 2. `notebookutils.credentials.getToken('kusto')` is the only Fabric-native
#    auth path that yields a token accepted by that endpoint (Fabric does
#    not support `DefaultAzureCredential`, and the token broker rejects
#    arbitrary resource URIs like `https://api.loganalytics.io/`).
# 3. The 'database' name passed to `KustoClient.execute` is the LAW resource
#    name (`log-ihzhhpf-sit`) — not the workspaceId GUID.
#
# See the Fabric portability note in the markdown cell above for the trail
# of alternatives we ruled out.

import notebookutils
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder

_sub = '66a9953a-df37-4c51-856c-9971b9bf3e03'
_rg = 'rg-ihzhhpf-sit'
law_cluster = (
    f'https://ade.loganalytics.io/subscriptions/{_sub}'
    f'/resourcegroups/{_rg}/providers/microsoft.operationalinsights'
    f'/workspaces/{log_analytics_workspace}'
)


def _fabric_kusto_token_provider() -> str:
    return notebookutils.credentials.getToken('kusto')


kcsb = KustoConnectionStringBuilder.with_token_provider(law_cluster, _fabric_kusto_token_provider)
client = KustoClient(kcsb)

result = client.execute(log_analytics_workspace, KQL)
table = result.primary_results[0]
columns = [c.column_name for c in table.columns]
raw_rows = [dict(zip(columns, row)) for row in table]

print(f'Fetched {len(raw_rows)} sign-in rows from {log_analytics_workspace} '
      f'(cluster={law_cluster}, lookback={lookback_hours}h)')


## 3. Load the persona → app-role roster

The Bronze contract carries `appRole` so the Sprint 15 BVA adoption KPI can
attribute each sign-in to a product capability. The roster mirrors
`data/synthetic/personas.csv`; an unknown user gets `appRole = None` and is
attributed to `Aggregated` downstream.


In [ ]:
persona_role = {}
try:
    roster = spark.read.option('header', True).csv(personas_path)
    persona_role = {
        r['upn'].lower(): r['app_role']
        for r in roster.select('upn', 'app_role').collect()
        if r['upn']
    }
except Exception as exc:  # roster is optional — Bronze stays raw, Silver can join later
    print(f'WARN: could not load personas roster from {personas_path}: {exc}')
print(f'Loaded {len(persona_role)} persona role mappings')


## 4. Map to the Bronze contract and land one JSON file per sign-in day

`adoption_transforms` is the pure, unit-tested mapping (see
`data-platform/notebooks/adoption/tests/`). It redacts the IP to a `/24`,
derives `env` from the app host, joins `appRole`, and normalises the timestamp.
Bronze is append-only and idempotent per day: re-running overwrites the same
`<day>/signins.json`; Silver handles deduplication across days.


In [ ]:
# --- BEGIN INLINED adoption_transforms.py ---
#
# Source of truth: data-platform/notebooks/adoption/adoption_transforms.py
# Inlined because Fabric notebooks can't `import` a repo-local .py module
# without uploading it as a Fabric Environment custom library.
# The module remains the tested unit (tests/test_adoption_transforms.py);
# this block is a copy that CI can lint for drift if we add an eq-check.
#
from collections.abc import Iterable, Mapping

# Canonical order of the Bronze adoption contract fields. Kept identical to the
# synthetic backfill output so real and seeded telemetry are interchangeable in
# Files/Bronze/adoption/YYYY-MM-DD/signins.json.
BRONZE_CONTRACT_FIELDS: tuple[str, ...] = (
    "userId",
    "upn",
    "appDisplayName",
    "appId",
    "signInTimestamp",
    "env",
    "resultType",
    "ipAddress",
    "clientAppUsed",
    "deviceDetailTrustType",
    "locationCountryOrRegion",
    "appRole",
)

# Successful sign-in result code (Entra). Downstream adoption KPIs count only
# successful sign-ins; Bronze keeps every row and lets Silver/Gold filter.
SIGNIN_SUCCESS = "0"

_PROD_SUFFIX = "-prod"


def redact_ip_24(ip: str | None) -> str | None:
    """Zero the last octet of an IPv4 address (``/24`` redaction).

    Non-IPv4 or empty values are returned unchanged (IPv6 has no ``/24`` octet
    notion and is passed through so Silver can decide how to bucket it).
    """
    if not ip:
        return ip
    octets = ip.split(".")
    if len(octets) != 4:
        return ip
    return f"{octets[0]}.{octets[1]}.{octets[2]}.0"


def derive_env(app_display_name: str | None, default: str = "sit") -> str:
    """Derive the deployment slot (``sit``/``prod``) from the app display name.

    The shared-user model (design spec §4) exposes one SIT slot and one PROD
    slot of the same app. PROD app registrations carry a ``-prod`` suffix; every
    other host resolves to ``sit`` during the demo phase.
    """
    name = (app_display_name or "").strip().lower()
    if name.endswith(_PROD_SUFFIX):
        return "prod"
    return default


def _iso_z(value) -> str:
    """Normalise a timestamp to ``...Z`` ISO-8601 form.

    Accepts ``str`` (Log Analytics/Graph already emit ISO) or anything with an
    ``isoformat`` method (e.g. ``datetime``). ``+00:00`` is normalised to ``Z``.
    """
    if value is None:
        return ""
    text = value.isoformat() if hasattr(value, "isoformat") else str(value)
    return text.replace("+00:00", "Z")


def signin_to_bronze_row(
    raw: Mapping,
    persona_role: Mapping[str, str] | None = None,
    default_env: str = "sit",
) -> dict:
    """Map one raw sign-in projection to a Bronze adoption contract row.

    ``raw`` uses the ``SigninLogs`` projection field names emitted by the KQL in
    ``01_adoption_ingest.ipynb`` (also produced by the Graph ``auditLogs/signIns``
    projection). ``persona_role`` maps a lower-cased UPN to its Entra app role so
    the adoption KPI can attribute each sign-in to a product capability; unknown
    users get ``appRole = None`` and are attributed to ``Aggregated`` downstream.
    """
    roles = persona_role or {}
    app_display_name = raw.get("AppDisplayName")
    upn = raw.get("UserPrincipalName")
    result_type = raw.get("ResultType")
    app_role = roles.get(upn.lower()) if isinstance(upn, str) and upn else None
    return {
        "userId": raw.get("UserId"),
        "upn": upn,
        "appDisplayName": app_display_name,
        "appId": raw.get("AppId"),
        "signInTimestamp": _iso_z(raw.get("TimeGenerated")),
        "env": derive_env(app_display_name, default=default_env),
        "resultType": str(result_type) if result_type is not None else None,
        "ipAddress": redact_ip_24(raw.get("IPAddress")),
        "clientAppUsed": raw.get("ClientAppUsed"),
        "deviceDetailTrustType": raw.get("DeviceDetail_TrustType"),
        "locationCountryOrRegion": raw.get("Location_CountryOrRegion"),
        "appRole": app_role,
    }


def to_bronze_rows(
    raw_rows: Iterable[Mapping],
    persona_role: Mapping[str, str] | None = None,
    default_env: str = "sit",
) -> list[dict]:
    """Map a batch of raw sign-in projections to Bronze contract rows."""
    return [signin_to_bronze_row(r, persona_role, default_env) for r in raw_rows]


def group_by_signin_day(rows: Iterable[Mapping]) -> dict[str, list[dict]]:
    """Group contract rows by their sign-in day (``YYYY-MM-DD``).

    Mirrors the synthetic backfill layout so each day lands as one
    ``Files/Bronze/adoption/<day>/signins.json`` file regardless of source.
    """
    by_day: dict[str, list[dict]] = {}
    for row in rows:
        ts = str(row.get("signInTimestamp", ""))
        if len(ts) < 10:
            continue
        by_day.setdefault(ts[:10], []).append(dict(row))
    return by_day
# --- END INLINED adoption_transforms.py ---

import json
import notebookutils

# `raw_rows` is populated in Cell 2 as list[dict] from Log Analytics via
# the ADX-compatible ADE endpoint (`kusto` audience token from Fabric).
bronze_rows = to_bronze_rows(raw_rows, persona_role, default_env=default_env)
by_day = group_by_signin_day(bronze_rows)

written = 0
for day, rows in sorted(by_day.items()):
    target = f'{bronze_root}/{day}/signins.json'
    payload = json.dumps(rows, indent=2, ensure_ascii=False)
    notebookutils.fs.put(target, payload, True)  # overwrite=True -- idempotent per day
    written += len(rows)
    print(f'wrote {len(rows):>5d} rows -> {target}')

print('-' * 72)
print(f'Bronze adoption ingest: {written} sign-in rows across {len(by_day)} day(s) under {bronze_root}/')
